# 01 — Análise exploratória

**CRISP-DM fase 2.3 — exploração dos dados** — Segmentação do mercado de veículos usados de Fortaleza

---

Este notebook percorre o **roteiro de oito etapas** da análise exploratória sobre a
base bruta de anúncios, usando o dicionário de dados montado em
[`00-dicionario-dados.ipynb`](00-dicionario-dados.ipynb) como mapa de tipos:

1. **Distribuição dos atributos** — o comportamento de cada característica analisada.
2. **Análise univariada** — distribuição de frequências e medidas de localização,
   dispersão, assimetria e curtose, uma variável por vez.
3. **Análise bivariada** — se a variação de uma característica é acompanhada pela
   variação de outra.
4. **Análise multivariada** — padrões e relações entre várias variáveis ao mesmo tempo.
5. **Valores ausentes** — onde falta dado e quanto essa falta pesa.
6. **Valores discrepantes** — registros cujo comportamento foge do esperado.
7. **Correlação** — qualquer relação estatística entre duas ou mais variáveis.
8. **Seleção de características** — a combinação de variáveis relevante para o
   aprendizado que vem depois.

Duas etapas deste roteiro não se aplicam aqui e são
tratadas à parte, no ponto em que apareceriam: **características ordinais** — esta
base não declara nenhuma escala ordinal — e **características de tempo** — a coleta
foi feita num único dia (`data_coleta` tem 1 valor distinto), então não há série
temporal para analisar.

Cada etapa grava as figuras em `reports/figures/eda-exploratoria/` e a tabela que
originou cada figura em `reports/tables/eda-exploratoria/` — nenhuma figura existe
sem a tabela que a produziu.

In [ ]:
# Importações
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import circlify
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import prince
import seaborn as sns

# O notebook roda a partir de `notebooks/`; a raiz entra no path para que
# `import src...` funcione sem instalar o pacote.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src import config
from src.data import dicionario as dic
from src.eda import estatistica as est
from src.utils import io

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

io.configurar_estilo()

TITULO = "SEGMENTAÇÃO DO MERCADO DE VEÍCULOS USADOS DE FORTALEZA"
SECAO = "eda-exploratoria"

BARRA = config.AZUL_COBALTO
REFERENCIA = config.AMARELO
DESTAQUE = config.AZUL_CLARO

#: Quantas categorias entram nos gráficos de frequência. `bairro` tem 182 níveis;
#: mostrar todos transforma o gráfico em lista.
TOPO = 10


def salvar(fig, nome, dados=None):
    '''Grava a figura em `reports/figures/eda-exploratoria/` e a tabela que a originou.'''
    return io.salvar_figura(fig, SECAO, nome, dados)


print(f"Projeto em {RAIZ}")

## Carregando a base de dados

Duas entradas: o **dicionário de dados** (`data/processed/dicionario.csv`, produzido
pelo notebook `00-dicionario-dados`) e a **base bruta**
(`data/raw/anuncios_detalhados.csv`).

Esta base não vem de um arquivo público único — é o resultado direto do scraping
(uma ferramenta separada, fora deste repositório). Por isso esta etapa também faz algo que outros roteiros de EDA
não precisam fazer: **mesclar o valor de FIPE**. A
OLX só traz a FIPE em 90,7% dos anúncios (`valor_fipe_olx`); os anúncios restantes
foram resolvidos à parte, via API pública da Tabela FIPE, com o resultado em
`data/raw/fipe_fallback.csv` (gerado pelo coletor, ver a página *Fonte dos dados*).

In [ ]:
# Carregando o dicionário de dados
dicionario = pd.read_csv(config.DATA_PROCESSED / "dicionario.csv", sep=";")

# Carregando a base bruta
bruto = pd.read_csv(config.ARQUIVO_BRUTO)

# Selecionando as mesmas colunas do dicionário de dados
bruto = bruto.loc[:, dicionario["Variável"]]

print(f"{bruto.shape[0]:,} linhas × {bruto.shape[1]} colunas")
bruto.head()

### Mesclando o valor de FIPE (OLX + API pública)

`fipe_fallback.csv` traz, para cada anúncio sem FIPE-OLX, o resultado da tentativa
de casar marca/modelo/ano com a API pública da Tabela FIPE por similaridade de
texto (casamento feito no coletor, ver a página *Fonte dos dados*). Nem toda tentativa funciona — ano fora da
tabela, modelo não identificado — então a cobertura sobe, mas não fecha em 100%.

In [ ]:
# Complemento de FIPE via API pública, para os anúncios sem FIPE-OLX
fallback = pd.read_csv(config.ARQUIVO_FIPE_FALLBACK)
# Um link aparece duas vezes no fallback (reprocessamento pontual); fica a última tentativa.
fallback_valor = fallback.drop_duplicates("link", keep="last").set_index("link")["valor_fipe_api"]

cobertura_olx = bruto["valor_fipe_olx"].notna().mean()

bruto["valor_fipe_final"] = bruto["valor_fipe_olx"].fillna(bruto["link"].map(fallback_valor))
bruto["fonte_fipe"] = np.select(
    [bruto["valor_fipe_olx"].notna(), bruto["valor_fipe_final"].notna()],
    ["OLX", "API pública (fallback)"],
    default="não encontrada",
)

cobertura_final = bruto["valor_fipe_final"].notna().mean()
print(f"Cobertura de FIPE: {cobertura_olx:.1%} (só OLX) → {cobertura_final:.1%} (OLX + API)")
bruto["fonte_fipe"].value_counts()

## O recorte da análise

Aqui o recorte é leve — a base bruta já tem só 23 colunas. Mesmo assim, nem
todas entram no roteiro
tipado abaixo:

* **identificador** (`link`) e **texto livre / alta cardinalidade**
  (`titulo`, `localizacao`, `versao`, `modelo` — 224 categorias, granularidade
  excessiva para os gráficos desta etapa) ficam de fora;
* `data_coleta` é metadado de coleta, não característica do veículo, e não varia
  (um único dia) — fora;
* `cor` e `motor` não estão no espaço de atributos declarado em
  `src/data/dicionario.py` (não constam entre as variáveis do Canvas do
  Problema) — ficam fora deste roteiro, mas continuam descritas no dicionário
  bruto.

Entram as **numéricas e nominais** do espaço de atributos
(`dic.NUMERICAS`, `dic.NOMINAIS`) e as três **reservadas**: `preco` (rótulo,
avaliação a posteriori), `bairro` (proxy de localização, validação externa) e
`valor_fipe_final` (derivada de marca/modelo/ano, leitura de oportunidade
comercial) — nenhuma das três entra na modelagem, mas todas passam pelo
roteiro exploratório completo.

In [ ]:
RECORTE = dic.NUMERICAS + dic.NOMINAIS + [dic.ROTULO_RESERVADO, dic.RESERVADA_VALIDACAO]

df = bruto.loc[:, RECORTE].copy()
df["valor_fipe_final"] = bruto["valor_fipe_final"]

# O dicionário acompanha o recorte, e a coluna derivada entra nele com o tipo herdado
# de `valor_fipe_olx` (Contínua).
dicionario = dicionario[dicionario["Variável"].isin(RECORTE)].reset_index(drop=True)
dicionario.loc[len(dicionario)] = {
    "Variável": "valor_fipe_final",
    "Tipo": "Contínua",
    "Valores/Faixa": [float(df["valor_fipe_final"].min()), float(df["valor_fipe_final"].max())],
    "Nulos": int(df["valor_fipe_final"].isna().sum()),
    "% nulos": round(100 * df["valor_fipe_final"].isna().mean(), 1),
    "Descrição": dic.DESCRICOES["valor_fipe_olx"],
}

print(f"{df.shape[1]} características analisadas, de {bruto.shape[1]} colunas na base bruta (já com FIPE mesclada)")
dicionario["Tipo"].value_counts()

Com o recorte fechado, cada tipo declarado no dicionário vira uma lista — é essa
lista que governa quais gráficos cada característica recebe nas etapas seguintes.
Esta base não tem colunas **ordinais** nem de **tempo** (ver nota na introdução),
então essas duas listas ficam vazias por construção.

In [ ]:
def por_tipo(tipo: str) -> list[str]:
    '''Nomes das colunas de um dado tipo semântico, na ordem do dicionário.'''
    return dicionario.query("Tipo == @tipo")["Variável"].tolist()


nominal_columns = por_tipo("Nominal")
continuous_columns = por_tipo("Contínua")
discrete_columns = por_tipo("Discreta")

# Contínuas e discretas recebem o mesmo tratamento estatístico: a distinção do
# dicionário é semântica, não computacional.
numericas = continuous_columns + discrete_columns

for nome, lista in [
    ("Nominais", nominal_columns),
    ("Contínuas", continuous_columns),
    ("Discretas", discrete_columns),
]:
    print(f"{nome:>10} ({len(lista):>2}): {', '.join(lista)}")

# 1ª etapa · Distribuição dos atributos

> *Na distribuição dos atributos é possível identificar o comportamento de cada uma das
> características analisadas.*

In [ ]:
def perfilar(dados: pd.DataFrame) -> dict:
    '''Contagens de topo do conjunto.'''
    quantitativas = dados.select_dtypes(include="number").columns
    celulas = dados.shape[0] * dados.shape[1]
    vazias = int(dados.isna().sum().sum())
    return {
        "Registros": dados.shape[0],
        "Características": dados.shape[1],
        "Numéricas": len(quantitativas),
        "Categóricas": dados.shape[1] - len(quantitativas),
        "Registros duplicados": int(dados.duplicated().sum()),
        "Células ausentes": vazias,
        "% de células ausentes": round(100 * vazias / celulas, 2),
    }


perfil_inicial = perfilar(df)

# Guardado para a 8ª etapa, que compara o conjunto de entrada com o selecionado.
df_inicial_qnt_feature = perfil_inicial["Características"]
df_inicial_qnt_feature_Numeric = perfil_inicial["Numéricas"]
df_inicial_qnt_feature_Categorical = perfil_inicial["Categóricas"]

pd.Series(perfil_inicial).to_frame(TITULO)

## Análises iniciais

Cada linha é um anúncio único (`link` já é a chave de deduplicação usada na coleta),
mas a checagem roda de novo aqui sobre o recorte, por precaução — duas linhas
idênticas nas características analisadas inflam qualquer média que vier depois.

In [ ]:
# Removendo registros duplicados
duplicados = int(df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)

perfil_final = perfilar(df)

panorama = pd.DataFrame({"Entrada": perfil_inicial, "Após deduplicação": perfil_final})
io.salvar_tabela(panorama.reset_index(names="medida"), SECAO, "panorama", indice=False)

print(f"{duplicados} registro(s) duplicado(s) removido(s)")
panorama

### Perfil coluna a coluna

Para cada característica: quantos valores distintos, quanto falta e — nas
numéricas — mínimo, mediana, máximo e a assimetria que decide se a variável
precisa de transformação na preparação.

In [ ]:
def descrever(dados: pd.DataFrame, dicionario: pd.DataFrame) -> pd.DataFrame:
    '''Perfil por coluna: cardinalidade, ausência e medidas de posição e forma.'''
    linhas = []
    for _, meta in dicionario.iterrows():
        coluna = dados[meta["Variável"]]
        quantitativa = pd.api.types.is_numeric_dtype(coluna)
        linhas.append(
            {
                "variavel": meta["Variável"],
                "tipo": meta["Tipo"],
                "distintos": int(coluna.nunique()),
                "ausentes": int(coluna.isna().sum()),
                "%ausentes": round(100 * coluna.isna().mean(), 2),
                "minimo": coluna.min() if quantitativa else "—",
                "mediana": coluna.median() if quantitativa else "—",
                "maximo": coluna.max() if quantitativa else "—",
                "assimetria": round(coluna.skew(), 2) if quantitativa else "—",
                "curtose": round(coluna.kurtosis(), 2) if quantitativa else "—",
            }
        )
    return pd.DataFrame(linhas)


perfil_colunas = descrever(df, dicionario)
io.salvar_tabela(perfil_colunas, SECAO, "perfil-das-colunas", indice=False)
perfil_colunas

# 2ª etapa · Análise univariada

> *A análise estatística univariada analisa a distribuição e a dispersão dos dados: a
> análise da distribuição de frequências e a análise das medidas de localização,
> dispersão, assimetria e curtose.*

## Características nominais

Frequência das categorias mais comuns, com o valor absoluto, a fração do total e a
linha tracejada na **frequência média por categoria**.

In [ ]:
for column in nominal_columns:
    contagem = df[column].value_counts()

    fig, ax = plt.subplots(figsize=(14, 6))
    sns.barplot(x=contagem.values[:TOPO], y=contagem.index[:TOPO], color=BARRA, ax=ax)

    total = float(contagem.sum())
    for grupo in ax.containers:
        larguras = [barra.get_width() for barra in grupo]
        ax.bar_label(
            grupo,
            label_type="edge",
            labels=[f"{largura:,.0f} · {largura / total:.1%}" for largura in larguras],
            fontsize=8,
            padding=3,
        )

    ax.axvline(contagem.values.mean(), color=REFERENCIA, ls="dashed", lw=2)

    ax.set_xlabel("Quantidade de registros")
    ax.set_ylabel(column)
    ax.set_title(f"Categorias mais frequentes de {column} ({contagem.size} no total)")
    ax.margins(x=0.15)

    salvar(fig, f"univariada-nominal-{column}", contagem.rename("registros").reset_index())

print(f"{len(nominal_columns)} figuras nominais gravadas")

## Características contínuas e discretas

Painel único por característica: extremos e amplitude, distribuição, dispersão
registro a registro e diagrama de caixa. O título traz a **assimetria**, a medida
que decide na fase de preparação quem recebe `log1p` — o projeto usa 0,75 como
limiar (`dic.LIMIAR_ASSIMETRIA`).

In [ ]:
def painel_numerico(column: str, familia: str) -> tuple:
    '''Extremos, distribuição, dispersão e caixa de uma característica numérica.'''
    serie = df[column].dropna()
    q1, q2, q3 = serie.quantile([0.25, 0.5, 0.75])

    fig, eixos = plt.subplots(2, 2, figsize=(14, 9))
    fig.suptitle(
        f"{column} — {familia} · assimetria {serie.skew():.2f}",
        fontsize=13,
        fontweight="bold",
        color=config.AZUL_PROFUNDO,
    )

    ax = eixos[0, 0]
    sns.barplot(x=["Mínimo", "Máximo"], y=[serie.min(), serie.max()], color=BARRA, ax=ax)
    for grupo in ax.containers:
        ax.bar_label(grupo, fmt="%.0f", fontsize=9, padding=3)
    ax.axhline(serie.max() - serie.min(), color=REFERENCIA, ls="dashed", lw=2)
    ax.set_title("Extremos e amplitude")
    ax.set_ylabel(column)

    ax = eixos[0, 1]
    sns.histplot(serie, bins=30, color=BARRA, ax=ax)
    ax.axvline(serie.mean(), color=REFERENCIA, ls="dashed", lw=2)
    ax.set_title(f"Distribuição · média {serie.mean():,.0f} · mediana {q2:,.0f}")
    ax.set_xlabel(column)

    ax = eixos[1, 0]
    sns.scatterplot(x=serie.index, y=serie.values, color=BARRA, s=12, alpha=0.5, ax=ax)
    ax.set_title("Dispersão por registro")
    ax.set_xlabel("Índice do registro")
    ax.set_ylabel(column)

    ax = eixos[1, 1]
    sns.boxplot(
        x=serie,
        color=BARRA,
        width=0.45,
        medianprops={"color": REFERENCIA, "linewidth": 2},
        ax=ax,
    )
    ax.set_title(f"Caixa · Q1 {q1:,.0f} · Q2 {q2:,.0f} · Q3 {q3:,.0f}")
    ax.set_xlabel(column)

    fig.tight_layout()

    resumo = pd.DataFrame(
        [
            {
                "variavel": column,
                "minimo": serie.min(),
                "q1": q1,
                "mediana": q2,
                "q3": q3,
                "maximo": serie.max(),
                "media": serie.mean(),
                "desvio": serie.std(),
                "assimetria": serie.skew(),
                "curtose": serie.kurtosis(),
            }
        ]
    )
    return fig, resumo


resumos = []
for column in continuous_columns:
    fig, resumo = painel_numerico(column, "característica contínua")
    salvar(fig, f"univariada-continua-{column}", resumo)
    resumos.append(resumo)

for column in discrete_columns:
    fig, resumo = painel_numerico(column, "característica discreta")
    salvar(fig, f"univariada-discreta-{column}", resumo)
    resumos.append(resumo)

resumo_numericas = pd.concat(resumos, ignore_index=True).sort_values(
    "assimetria", ascending=False
)
io.salvar_tabela(resumo_numericas, SECAO, "univariada-resumo-numericas", indice=False)
resumo_numericas

# 3ª etapa · Análise bivariada

> *Na avaliação estatística da causalidade, convencionou-se descrever essas análises
> bivariadas como formas de investigar se a variação de uma característica pode ser
> explicada pela variação de outra medida.*

O cruzamento parte de **quatro eixos de leitura** — os recortes pelos quais uma
revenda organiza o mercado (marca, carroceria, câmbio, tipo de vendedor) — contra
**quatro medidas** de preço, uso e referência. Cada barra traz a média do grupo e
quanto ela representa da média geral, marcada na linha tracejada.

## Nominais × contínuas e discretas

In [ ]:
#: Os recortes pelos quais o negócio lê o mercado.
EIXOS = ["marca", "carroceria", "cambio", "vendedor_tipo"]

#: As medidas que o Canvas do Problema nomeia: preço, uso e referência de mercado.
MEDIDAS_CONTINUAS = ["valor_fipe_final", "ano"]
MEDIDAS_DISCRETAS = ["preco", "km"]


def bivariada_barras(categoria: str, medida: str, familia: str) -> pd.DataFrame:
    '''Média da medida por categoria, ordenada, com a média geral como referência.'''
    tabela = (
        df.groupby(categoria, observed=True)[medida]
        .agg(media="mean", registros="size")
        .reset_index()
        .sort_values("media", ascending=False)
    )
    recorte = tabela.head(TOPO)
    media_geral = df[medida].mean()

    fig, ax = plt.subplots(figsize=(14, 6))
    sns.barplot(x=recorte["media"], y=recorte[categoria], color=BARRA, ax=ax)
    for grupo in ax.containers:
        larguras = [barra.get_width() for barra in grupo]
        ax.bar_label(
            grupo,
            label_type="edge",
            labels=[f"{v:,.0f} · {v / media_geral:.0%}" for v in larguras],
            fontsize=8,
            padding=3,
        )
    ax.axvline(media_geral, color=REFERENCIA, ls="dashed", lw=2)
    ax.set_xlabel(f"Média de {medida}")
    ax.set_ylabel(categoria)
    ax.set_title(f"{categoria} × média de {medida} — característica {familia}")
    ax.margins(x=0.18)

    salvar(fig, f"bivariada-{medida} X {categoria}", tabela)
    return tabela


for categoria in EIXOS:
    for medida in MEDIDAS_CONTINUAS:
        bivariada_barras(categoria, medida, "contínua")
    for medida in MEDIDAS_DISCRETAS:
        bivariada_barras(categoria, medida, "discreta")

print(f"{len(EIXOS) * (len(MEDIDAS_CONTINUAS) + len(MEDIDAS_DISCRETAS))} figuras gravadas")

# 4ª etapa · Análise multivariada

> *Análise multivariada é uma ferramenta que encontra padrões e relações entre várias
> variáveis, permitindo prever efeitos e mudanças que uma variável terá sobre a outra.*

Duas variáveis nominais não têm correlação de Pearson — não têm ordem para
correlacionar. A **análise de correspondência** (`prince.CA`) resolve isso: decompõe
a tabela de contingência e projeta linhas e colunas no mesmo plano, onde a
proximidade entre uma marca e uma carroceria significa que aquela combinação
aparece mais do que o acaso explicaria.

Cada categoria de linha é então vinculada à categoria de coluna **mais próxima** no
plano, e o resultado vai para um empacotamento de círculos: o círculo escuro é o
grupo, os claros dentro dele são as categorias que gravitam em torno daquele grupo.

O critério de corte é a **inércia acumulada nas duas dimensões** (`percentage_of_variance_`
no `prince` ≥ 0.20): um plano só é legível se as duas dimensões juntas retiverem o
suficiente da tabela original. O piso é 80%.

In [ ]:
#: Nominais com cardinalidade que sustenta uma tabela de contingência estável.
NOMINAIS_CA = ["marca", "carroceria", "cambio", "combustivel"]

#: Inércia acumulada mínima nas duas dimensões para que o plano seja interpretável.
INERCIA_MINIMA = 80.0


def distancia_euclidiana(px, qx, py, qy):
    '''Distância entre dois pontos do plano fatorial.'''
    return float(np.sqrt((px - qx) ** 2 + (py - qy) ** 2))


def circulos_do_agrupamento(vinculos: pd.DataFrame) -> list:
    '''Hierarquia total → grupo → categoria, no formato que o circlify espera.'''
    dados = [
        {
            "id": "Total",
            "datum": len(vinculos),
            "children": [
                {
                    "id": grupo,
                    "datum": int((vinculos.grupo == grupo).sum()),
                    "children": [
                        {"id": categoria, "datum": 1}
                        for categoria in vinculos.query("grupo == @grupo").categoria
                    ],
                }
                for grupo in vinculos.grupo.unique()
            ],
        }
    ]
    return circlify.circlify(
        dados, show_enclosure=False, target_enclosure=circlify.Circle(x=0, y=0, r=1)
    )


inercias = []

for p in range(len(NOMINAIS_CA)):
    for k in range(p + 1, len(NOMINAIS_CA)):
        primeira, segunda = NOMINAIS_CA[p], NOMINAIS_CA[k]

        contingencia = pd.crosstab(df[primeira], df[segunda])
        if contingencia.shape[0] < contingencia.shape[1]:
            contingencia = contingencia.T
        if min(contingencia.shape) < 2:
            continue

        modelo = prince.CA(n_components=2, random_state=config.SEMENTE).fit(contingencia)
        inercia = float(np.sum(modelo.percentage_of_variance_))
        inercias.append(
            {
                "linhas": contingencia.index.name,
                "colunas": contingencia.columns.name,
                "inercia_2d": round(inercia, 2),
                "figura": inercia >= INERCIA_MINIMA,
            }
        )
        if inercia < INERCIA_MINIMA:
            continue

        linhas = modelo.row_coordinates(contingencia)
        colunas = modelo.column_coordinates(contingencia)
        pares = []
        for i, categoria in enumerate(contingencia.index):
            distancias = [
                distancia_euclidiana(
                    linhas.iloc[i, 0], colunas.iloc[j, 0], linhas.iloc[i, 1], colunas.iloc[j, 1]
                )
                for j in range(len(colunas))
            ]
            pares.append(
                {
                    "categoria": categoria,
                    "grupo": contingencia.columns[int(np.argmin(distancias))],
                    "distancia": round(min(distancias), 3),
                }
            )
        vinculos = pd.DataFrame(pares)
        if vinculos.grupo.nunique() < 2:
            continue

        circulos = circulos_do_agrupamento(vinculos)

        fig, ax = plt.subplots(figsize=(11, 11))
        ax.axis("off")
        ax.set_aspect("equal")
        limite = max(max(abs(c.x) + c.r, abs(c.y) + c.r) for c in circulos)
        ax.set_xlim(-limite, limite)
        ax.set_ylim(-limite, limite)

        for circulo in circulos:
            if circulo.level == 2:
                ax.add_patch(plt.Circle((circulo.x, circulo.y), circulo.r, color=BARRA))

        for circulo in circulos:
            if circulo.level == 3:
                ax.add_patch(
                    plt.Circle((circulo.x, circulo.y), circulo.r, alpha=0.55, color=DESTAQUE)
                )
                ax.annotate(
                    circulo.ex["id"],
                    (circulo.x, circulo.y),
                    ha="center",
                    va="center",
                    fontsize=8,
                    color="#12303F",
                )

        for circulo in circulos:
            if circulo.level == 2:
                ax.annotate(
                    circulo.ex["id"],
                    (circulo.x, circulo.y + circulo.r * 0.82),
                    ha="center",
                    va="center",
                    fontsize=9,
                    color="white",
                    bbox=dict(
                        facecolor=config.AZUL_PROFUNDO, edgecolor="none", boxstyle="round,pad=0.4"
                    ),
                )

        ax.set_title(
            f"{contingencia.index.name} agrupada por {contingencia.columns.name}"
            f" — inércia acumulada {inercia:.1f}%"
        )
        salvar(fig, f"multivariada-ca-{primeira} X {segunda}", vinculos)

inercias = pd.DataFrame(inercias).sort_values("inercia_2d", ascending=False)
io.salvar_tabela(inercias, SECAO, "multivariada-inercia-dos-pares", indice=False)
inercias

# 5ª etapa · Detecção de valores ausentes

> *Detecção de valores ausentes é uma tarefa comum na área de ciência de dados para
> mensurar os impactos dessa ausência no conjunto de dados em estudo.*

A contagem roda sobre a **base bruta completa** (23 colunas + `valor_fipe_final`
mesclada), não só o recorte.

Aqui **não há ausência estrutural** (nenhuma coluna
onde "vazio" signifique "o veículo não tem esse item"): toda ausência é real —
o vendedor não informou, ou o dado não pôde ser resolvido (caso de
`valor_fipe_final`, mesmo após o fallback pela API).

In [ ]:
ausentes = (
    bruto.isna()
    .sum()
    .rename("ausentes")
    .to_frame()
    .assign(percentual=lambda t: (100 * t.ausentes / len(bruto)).round(2))
    .query("ausentes > 0")
    .sort_values("ausentes", ascending=False)
    .reset_index(names="variavel")
)

fig, ax = plt.subplots(figsize=(14, 6))
sns.barplot(data=ausentes, x="variavel", y="percentual", color=BARRA, ax=ax)
for grupo in ax.containers:
    ax.bar_label(grupo, fmt="%.1f%%", fontsize=8, padding=2)
ax.tick_params(axis="x", rotation=60)
ax.set_xlabel("Características com ao menos uma célula vazia")
ax.set_ylabel("% de células ausentes")
ax.set_title(
    f"Ausência por coluna — {len(ausentes)} de {bruto.shape[1]} características afetadas"
)

salvar(fig, "valores-ausentes", ausentes)
ausentes

# 6ª etapa · Detecção de valores discrepantes

> *A detecção de outliers consiste na identificação de padrões em dados com um
> comportamento diferente do esperado. Estes padrões são muitas vezes referidos como
> anomalias.*

O critério é o de Tukey — fora do intervalo `[Q1 − 1,5·IQR, Q3 + 1,5·IQR]`
(`est.outliers_iqr`). Discrepante não é erro: um carro de 999.998 km existe; é a
revenda que precisa decidir se ele pertence à mesma prateleira dos outros.

In [ ]:
linhas = []
for column in numericas:
    serie = df[column].dropna()
    marca = est.outliers_iqr(serie)
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    linhas.append(
        {
            "variavel": column,
            "discrepantes": int(marca.sum()),
            "percentual": round(100 * marca.mean(), 2),
            "limite_inferior": q1 - 1.5 * iqr,
            "limite_superior": q3 + 1.5 * iqr,
            "menor_discrepante": serie[marca].min() if marca.any() else np.nan,
            "maior_discrepante": serie[marca].max() if marca.any() else np.nan,
        }
    )

discrepantes = pd.DataFrame(linhas).sort_values("percentual", ascending=False)

fig, ax = plt.subplots(figsize=(14, 6))
sns.barplot(data=discrepantes, x="percentual", y="variavel", color=BARRA, ax=ax)
for grupo in ax.containers:
    ax.bar_label(grupo, fmt="%.2f%%", fontsize=8, padding=3)
ax.axvline(discrepantes.percentual.mean(), color=REFERENCIA, ls="dashed", lw=2)
ax.set_xlabel("% de registros fora das cercas de Tukey")
ax.set_ylabel("")
ax.set_title("Discrepantes por característica — critério IQR (1,5×)")
ax.margins(x=0.15)

salvar(fig, "valores-discrepantes", discrepantes)
discrepantes

# 7ª etapa · Detecção de correlação

> *Em probabilidade e estatística, correlação, dependência ou associação é qualquer
> relação estatística entre duas ou mais variáveis.*

Três medidas conforme o par de tipos, todas em `src/eda/estatistica.py`:

| Par de tipos | Medida | Por quê |
|:---|:---|:---|
| numérica × numérica | Spearman | monótona, não presume linearidade nem normalidade |
| nominal × nominal | V de Cramér | qui-quadrado normalizado, com correção de Bergsma |
| nominal × numérica | η (razão de correlação) | fração da variância que a categoria explica |

Esta base não tem ordinais, então não há etapa de codificação por escala antes da
matriz — todas as numéricas entram como estão.

In [ ]:
quantitativas = numericas
qualitativas = nominal_columns
colunas = quantitativas + qualitativas

correlacao = pd.DataFrame(np.eye(len(colunas)), index=colunas, columns=colunas)
for i, primeira in enumerate(colunas):
    for segunda in colunas[i + 1 :]:
        if primeira in quantitativas and segunda in quantitativas:
            valor = df[primeira].corr(df[segunda], method="spearman")
        elif primeira in qualitativas and segunda in qualitativas:
            valor = est.v_de_cramer(df[primeira], df[segunda])
        else:
            categorica, quantitativa = (
                (primeira, segunda) if primeira in qualitativas else (segunda, primeira)
            )
            valor = np.sqrt(est.razao_correlacao(df[categorica], df[quantitativa]))
        correlacao.loc[primeira, segunda] = correlacao.loc[segunda, primeira] = abs(float(valor))

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(
    correlacao,
    cmap="Blues",
    vmin=0,
    vmax=1,
    annot=True,
    fmt=".2f",
    annot_kws={"size": 7},
    square=True,
    cbar_kws={"shrink": 0.7},
    ax=ax,
)
ax.set_title("Associação entre características — Spearman, V de Cramér e η")

salvar(fig, "correlacao", correlacao.reset_index(names="variavel"))

# Pares acima do limiar de redundância: a mesma informação medida duas vezes.
redundantes = est.pares_redundantes(correlacao, limiar=0.7)
io.salvar_tabela(redundantes, SECAO, "pares-redundantes", indice=False)
redundantes

# 8ª etapa · Seleção de características

> *Os métodos de seleção de características têm a missão de encontrar uma combinação
> adequada de características relevantes para o aprendizado de um conceito.*

Cada variável recebe uma decisão, com o motivo ao lado — é o que a fase de
preparação (notebook `03-preparacao`) recebe como entrada:

* **reservada** — `preco` é o rótulo da avaliação a posteriori, `bairro` é proxy de
  localização (reservada para validação externa) e `valor_fipe_final` é derivada de
  marca/modelo/ano (reservada para a leitura de oportunidade comercial). Nenhuma das
  três entra na segmentação;
* **descartada por redundância** — dentro de cada par acima de 0,7 sai a medida de
  **menor resolução** (menos valores distintos). Empate de cardinalidade se resolve
  pela maior associação média com o resto. O critério **não olha para `preco`**;
* **mantida** — o resto.

In [ ]:
df_inicial = pd.DataFrame(
    [
        {
            "Quantidade de características": df_inicial_qnt_feature,
            "Características numéricas": df_inicial_qnt_feature_Numeric,
            "Características categóricas": df_inicial_qnt_feature_Categorical,
        }
    ]
)
io.salvar_tabela(df_inicial, SECAO, "df_inicial", indice=False)

RESERVADAS = {
    "preco": "rótulo reservado para a avaliação a posteriori",
    "bairro": "proxy de localização, reservada para validação externa",
    "valor_fipe_final": "derivada de marca/modelo/ano, reservada para leitura de oportunidade",
}

decisoes = {variavel: ("reservada", motivo) for variavel, motivo in RESERVADAS.items()}

for _, par in redundantes.iterrows():
    a, b = par.variavel_a, par.variavel_b
    if a in decisoes or b in decisoes:
        continue
    if df[a].nunique() != df[b].nunique():
        descartada, mantida = (a, b) if df[a].nunique() < df[b].nunique() else (b, a)
        criterio = f"{df[descartada].nunique()} valores distintos contra {df[mantida].nunique()}"
    else:
        ativas = [c for c in correlacao.columns if c not in decisoes]
        media_a = correlacao.loc[a, ativas].drop(a).mean()
        media_b = correlacao.loc[b, ativas].drop(b).mean()
        descartada, mantida = (a, b) if media_a > media_b else (b, a)
        criterio = "mesma cardinalidade; sai a de maior associação média"
    decisoes[descartada] = (
        "descartada",
        f"associação {par.correlacao:.2f} com {mantida} · {criterio}",
    )

tipos = dicionario.set_index("Variável")["Tipo"]
selecao = pd.DataFrame(
    [
        {
            "variavel": variavel,
            "tipo": tipos.get(variavel, "—"),
            "decisao": decisoes.get(variavel, ("mantida", "sem redundância acima do limiar"))[0],
            "motivo": decisoes.get(variavel, ("mantida", "sem redundância acima do limiar"))[1],
        }
        for variavel in correlacao.columns
    ]
).sort_values(["decisao", "variavel"])

io.salvar_tabela(selecao, SECAO, "selecao-de-caracteristicas", indice=False)

print(
    f"{(selecao.decisao == 'mantida').sum()} mantidas · "
    f"{(selecao.decisao == 'descartada').sum()} descartadas por redundância · "
    f"{(selecao.decisao == 'reservada').sum()} reservadas"
)
selecao

# Insights

O que as oito etapas deixaram, com o número que sustenta cada afirmação. É esta
lista que a fase de preparação (`02-ajustes-dados`) consome — cada item ou vira uma
decisão de tratamento, ou vira uma restrição que a modelagem precisa respeitar.

In [ ]:
insights = [
    "`km` tem um registro de 999.998 — muito acima do limite superior de Tukey "
    "(265.000) e visivelmente um valor de preenchimento/erro de digitação, não um "
    "carro real. Candidato a tratamento na 02-ajustes-dados.",
    "`preco` e `valor_fipe_final` correlacionam em 0,98 (Spearman) — a FIPE mesclada "
    "é, por construção, quase um espelho do preço pedido. Confirma por que ela fica "
    "reservada da modelagem (redundante com o próprio preço) e não descartada por "
    "correlação: ambas já saem por serem reservadas.",
    "Câmbio separa o mercado com força mesmo antes de qualquer clusterização: "
    "automático custa em média R$ 120 mil com 68 mil km rodados, contra R$ 42,5 mil "
    "e 113 mil km do manual — dois perfis de veículo bem diferentes.",
    "O desconto médio sobre a FIPE varia por marca: Peugeot (13,3%), Citroën (8,3%) "
    "e Chery (8,2%) no topo; BYD é a única marca com desconto negativo (-9,3%, "
    "anunciada acima da FIPE) — primeira leitura de oportunidade comercial "
    "(Pergunta 5 do projeto).",
    "Lojas profissionais concentram veículos com menor km (74 mil contra 130 mil de "
    "particulares), mas desconto médio negativo sobre a FIPE (-2,0%); particulares "
    "vendem com desconto médio positivo (+1,9%) — menor km e melhor preço vs. FIPE "
    "não andam juntos no mesmo canal de venda.",
    "`aceita_troca` (27,4%) e `unico_dono` (18,7%), medidos na base bruta de 2.565 "
    "anúncios, são as colunas com mais ausência "
    "— esperado, dependem do anunciante informar (ver a página *Fonte dos dados* "
    "da documentação). "
    "A mescla com a API pública reduz a ausência de FIPE de 9,3% para 2,9%.",
]

insights = pd.DataFrame(insights, columns=["insights"])
io.salvar_tabela(insights, SECAO, "insights", indice=False)

for numero, texto in enumerate(insights.insights, start=1):
    print(f"{numero:>2}. {texto}\n")

## Próximo passo

As oito etapas deste roteiro descrevem a base **como ela chega**, já com a FIPE
mesclada. O notebook [`02-ajustes-dados`](02-ajustes-dados.ipynb) aplica as decisões
de limpeza (duplicados, discrepantes, ausência) e produz a base depurada em
`data/processed/`.

As figuras deste notebook estão em `reports/figures/eda-exploratoria/` e as tabelas
em `reports/tables/eda-exploratoria/`.